In [ ]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("stats_file.xlsx")
daily = pd.read_excel("daily_demand.xlsx")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle_Time"]) * data["Cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand_tomorrow = row["Actual_Tomorrow"]
    tentative_future = row["Tentative_Future"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    for qty in np.arange(0, 5000, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

In [ ]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40  # minutes
SETUP_HOURS = SETUP_TIME / 60

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["Sum of 28-02-2026"]
    tentative_future = row["Sum of 02-03-2026"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    best_qty = 0
    best_cost = np.inf

    max_qty = row["Feb INDENT"] * 1.2

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.random.normal(
            tentative_future, std, SIMULATIONS
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        # Setup penalty (only if production happens)
        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        hours_needed = qty / rate

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

# ============================
# TIME REQUIRED COLUMN
# ============================

data["Time Required (hrs)"] = data["Planned_Qty"] / data["Rate"]

# ============================
# SAVE OUTPUT
# ============================

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

In [ ]:
import pandas as pd
import numpy as np

# ============================
# PARAMETERS
# ============================

SIMULATIONS = 1000
SHORTAGE_PENALTY = 10
HOLDING_PENALTY = 1
AVAILABLE_HOURS = 22

SETUP_TIME = 40  # minutes
SETUP_HOURS = SETUP_TIME / 60

# ============================
# LOAD FILES
# ============================

stats = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="Sheet2")
daily = pd.read_excel("C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx", sheet_name="Sheet1")

data = stats.merge(daily, on="Material")

# ============================
# PRODUCTION RATE
# ============================

data["Rate"] = (3600 / data["Cycle time "]) * data["cavity"]

# ============================
# OPTIMIZATION FUNCTION
# ============================

def optimize_part(row):

    inventory = row["Inventory on 24th"]
    demand_tomorrow = row["Sum of 28-02-2026"]
    tentative_future = row["Sum of 02-03-2026"]
    std = row["Std_Deviation"]
    rate = row["Rate"]
    indent = row["Feb INDENT"]

    best_qty = 0
    best_cost = np.inf

    # Avoid overproduction
    target_stock = demand_tomorrow + tentative_future + std
    max_qty = max(0, target_stock - inventory)
    max_qty = min(max_qty, indent * 1.2)

    for qty in np.arange(0, max_qty + 1, 50):

        future_stock = inventory + qty - demand_tomorrow

        simulated_demand = np.maximum(
            0,
            np.random.normal(tentative_future, std, SIMULATIONS)
        )

        shortage = np.maximum(0, simulated_demand - future_stock)

        expected_shortage = shortage.mean()
        expected_inventory = max(0, future_stock - simulated_demand.mean())

        setup_penalty = rate * SETUP_HOURS if qty > 0 else 0

        cost = (
            SHORTAGE_PENALTY * expected_shortage +
            HOLDING_PENALTY * expected_inventory +
            setup_penalty
        )

        # Protect tomorrow
        tomorrow_shortage = max(0, demand_tomorrow - (inventory + qty))
        cost += SHORTAGE_PENALTY * tomorrow_shortage * 2

        # Setup consumes time
        hours_needed = (qty / rate + SETUP_HOURS) if qty > 0 else 0

        if hours_needed <= AVAILABLE_HOURS and cost < best_cost:
            best_cost = cost
            best_qty = qty

    # ================================
    # STRATEGIC CLOSE LOGIC
    # ================================

    full_time = indent / rate + SETUP_HOURS

    if full_time <= AVAILABLE_HOURS:

        remaining = max(0, indent - (inventory + best_qty))

        if tentative_future > 0:
            future_runs = remaining / tentative_future
        else:
            future_runs = 0

        setup_saving = future_runs * rate * SETUP_HOURS

        strategic_bonus = setup_saving * 0.3

        full_cost = HOLDING_PENALTY * max(0, indent - (inventory + demand_tomorrow))

        adjusted_full_cost = full_cost - strategic_bonus

        if adjusted_full_cost < best_cost:
            best_qty = indent
            best_cost = adjusted_full_cost

    return best_qty, best_cost

# ============================
# RUN OPTIMIZATION
# ============================

results = data.apply(optimize_part, axis=1)

data["Planned_Qty"] = [r[0] for r in results]
data["Cost"] = [r[1] for r in results]

# ============================
# TIME REQUIRED COLUMN
# ============================

data["Time Required (hrs)"] = np.where(
    data["Planned_Qty"] > 0,
    (data["Planned_Qty"] / data["Rate"]) + SETUP_HOURS,
    0
)

# ============================
# SAVE OUTPUT
# ============================

data.to_excel("daily_plan.xlsx", index=False)

print("✅ Optimization complete")

In [ ]:
import pandas as pd
import numpy as np

# ============================
# LOAD EXCEL FILES
# ============================

fileA = pd.read_excel("planned_qty.xlsx", sheet_name="Sheet1")
fileB = pd.read_excel("production_data.xlsx", sheet_name="Sheet1")

# ============================
# MERGE DATA
# ============================

data = fileA.merge(fileB, on="part_id", how="left")
data.fillna(0, inplace=True)

# ============================
# PLAN COMPARISON
# ============================

def compare(row):
    if row["their_planned_qty"] > row["our_planned_qty"]:
        return "Higher than ours"
    elif row["their_planned_qty"] < row["our_planned_qty"]:
        return "Lower than ours"
    else:
        return "Equal"

data["Plan_Comparison"] = data.apply(compare, axis=1)

# ============================
# DOWNTIME IMPACT
# ============================

data["Downtime_Impact"] = (
    (data["Plan_Comparison"] == "Lower than ours") &
    (data["downtime"] > 0)
)

# ============================
# OUR PLAN vs PRODUCED
# ============================

data["Production_Gap"] = data["produced_qty"] - data["our_planned_qty"]

# ============================
# INDENT COMPLETION
# ============================

data["Indent_Gap"] = data["produced_qty"] - data["indent_qty"]
data["Complete_Indent"] = data["produced_qty"] >= data["indent_qty"]

# ============================
# QUALITY
# ============================

data["Rejection_Rate"] = np.where(
    data["produced_qty"] == 0,
    0,
    data["rej"] / data["produced_qty"]
)

# ============================
# CYCLE PERFORMANCE
# ============================

data["Cycle_Efficiency"] = np.where(
    data["actual_cycle_time"] == 0,
    0,
    data["target_cycle_time"] / data["actual_cycle_time"]
)

# ============================
# TIME LOSS
# ============================

data["Total_Loss_Time"] = data["downtime"] + data["break_time"]

# ============================
# MACHINE SUMMARY
# ============================

machine_summary = (
    data.groupby("machine_id")
    .agg({
        "produced_qty": "sum",
        "downtime": "sum",
        "break_time": "sum"
    })
    .reset_index()
)

machine_summary.rename(columns={
    "produced_qty": "Machine_Production",
    "downtime": "Total_Downtime",
    "break_time": "Total_Break_Time"
}, inplace=True)

# ============================
# SAVE EXCEL REPORT
# ============================

with pd.ExcelWriter("comparison_report.xlsx") as writer:
    data.to_excel(writer, sheet_name="Part_Comparison", index=False)
    machine_summary.to_excel(writer, sheet_name="Machine_Summary", index=False)

# ============================
# DASHBOARD DATA
# ============================

dashboard = data[[
    "part_id",
    "our_planned_qty",
    "their_planned_qty",
    "produced_qty",
    "indent_qty",
    "Plan_Comparison",
    "Downtime_Impact",
    "Production_Gap",
    "Complete_Indent",
    "Rejection_Rate",
    "Cycle_Efficiency",
    "Total_Loss_Time",
    "machine_id"
]]

dashboard.to_excel("dashboard_data.xlsx", index=False)

print("✅ Excel Report Generated")
print("✅ Dashboard Dataset Ready")